# SQL

In [1]:
GITHUB_DB_URL = "https://github.com/siara-cc/sakila_sqlite3/raw/master/sakila.db"

In [2]:
import os
import urllib.request
import sqlite3
import pandas as pd

# Nom du fichier local
DB_FILENAME = "sakila.db"

# URL brute corrigée pour télécharger le fichier .db
GITHUB_DB_URL = "https://github.com/siara-cc/sakila_sqlite3/raw/master/sakila.db"

# Si le fichier existe déjà, on le supprime (au cas où il serait corrompu)
if os.path.exists(DB_FILENAME):
    os.remove(DB_FILENAME)

# Téléchargement de la base depuis GitHub
print("Téléchargement de la base SQLite...")
urllib.request.urlretrieve(GITHUB_DB_URL, DB_FILENAME)
print("Téléchargement terminé.")

# Création de la fonction helper pour exécuter des requêtes SQL
def run_query(query):
    conn = sqlite3.connect(DB_FILENAME)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


Téléchargement de la base SQLite...
Téléchargement terminé.


Schema de sakila :
![](https://www.jooq.org/img/sakila.png)

In [3]:
"""
📌 Question 1 :
Afficher les 10 films les plus longs avec :
- leur titre,
- leur durée,
- leur catégorie,
- un acteur associé à chaque film.

Tables utiles : film, film_category, category, film_actor, actor
"""

query1 = """
tapez ici la requete
"""

run_query(query1)


DatabaseError: Execution failed on sql '
tapez ici la requete
': near "tapez": syntax error

In [4]:
# QUESTION 1

query1 = """
SELECT film.title, film.length, category.name as category, actor.first_name ||" " || actor.last_name as actor_name
FROM film
INNER JOIN film_category USING(film_id)
INNER JOIN category USING(category_id)
INNER JOIN film_actor USING(film_id)
INNER JOIN actor USING(actor_id)
ORDER BY film.length DESC
LIMIT 10;
"""

run_query(query1)

,title,length,category,actor_name
0,SOLDIERS EVOLUTION,185,Sci-Fi,JOHNNY LOLLOBRIGIDA
1,SOLDIERS EVOLUTION,185,Sci-Fi,UMA WOOD
2,SOLDIERS EVOLUTION,185,Sci-Fi,CUBA OLIVIER
3,CONTROL ANTHEM,185,Comedy,BOB FAWCETT
4,DARN FORRESTER,185,Action,BOB FAWCETT
5,HOME PITY,185,Music,KIRSTEN PALTROW
6,GANGS PRIDE,185,Animation,ELVIS MARX
7,DARN FORRESTER,185,Action,SANDRA KILMER
8,CONTROL ANTHEM,185,Comedy,AUDREY OLIVIER
9,SOLDIERS EVOLUTION,185,Sci-Fi,JUDY DEAN


In [5]:
"""
📌 Question 2 :
Afficher les 10 clients ayant effectué le plus de locations :
- leur nom complet,
- le nombre total de locations,
- leur classement (ROW_NUMBER),
- le cumul progressif du nombre de locations (fonction SUM OVER).

Tables : rental, customer

HINT : utiliser ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW dans le sum over pour eviter de comptabiliser les total rentals egaux dans la somme cumulée
"""

query2 = """
tapez ici la requete
"""

run_query(query2)


DatabaseError: Execution failed on sql '
tapez ici la requete
': near "tapez": syntax error

In [6]:
query2 = """
SELECT customer.first_name ||" " || customer.last_name as full_name,
COUNT(rental_id) as total_rentals,
ROW_NUMBER() OVER(ORDER BY COUNT(rental_id) DESC) as rank,
SUM(COUNT(rental_id)) OVER(ORDER BY COUNT(rental_id) DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as cumulative_rentals
FROM customer
INNER JOIN rental USING(customer_id)
GROUP BY full_name
LIMIT 10
"""

run_query(query2)

,full_name,total_rentals,rank,cumulative_rentals
0,ELEANOR HUNT,46,1,46
1,KARL SEAL,45,2,91
2,MARCIA DEAN,42,3,133
3,CLARA SHAW,42,4,175
4,TAMMY SANDERS,41,5,216
5,WESLEY BULL,40,6,256
6,SUE PETERS,40,7,296
7,TIM CARY,39,8,335
8,RHONDA KENNEDY,39,9,374
9,MARION SNYDER,39,10,413


In [ ]:
"""
📌 Question 3 :
Afficher l'évolution mensuelle du nombre de films loués pour les 3 catégories les plus populaires.
Pour chaque mois et chaque catégorie :
- le nombre de films loués,
- le nombre du mois précédent,
- la variation (différence).

Tables : rental, inventory, film, film_category, category
Fonction utilisée : LAG (fonction fenêtre)
"""

query3 = """
tapez ici la requete
"""

run_query(query3)

,rental_month,category,rental_count,previous_month_count,variation
0,2005-05,Action,87,NaN,NaN
1,2005-06,Action,160,87.0,73.0
2,2005-07,Action,464,160.0,304.0
3,2005-08,Action,384,464.0,-80.0
4,2006-02,Action,17,384.0,-367.0
5,2005-05,Animation,74,NaN,NaN
6,2005-06,Animation,174,74.0,100.0
7,2005-07,Animation,489,174.0,315.0
8,2005-08,Animation,408,489.0,-81.0
9,2006-02,Animation,21,408.0,-387.0


In [7]:
# Je trouve les catégories les plus populaires

querytest = """
SELECT category.name as category, COUNT(rental_id)
FROM rental
JOIN inventory USING(inventory_id)
JOIN film USING(film_id)
JOIN film_category USING(film_id)
JOIN category USING(category_id)
GROUP BY category.name
ORDER BY COUNT(rental_id) DESC LIMIT 3
"""

run_query(querytest)

,category,COUNT(rental_id)
0,Sports,1179
1,Animation,1166
2,Action,1112


In [8]:
query3 = """
WITH CTE as
  (SELECT STRFTIME('%Y-%m', rental_date) as rental_month,
  category.name as category,
  COUNT(rental_id) as rental_count
FROM rental
JOIN inventory USING(inventory_id)
JOIN film USING(film_id)
JOIN film_category USING(film_id)
JOIN category USING(category_id)
WHERE category.name IN("Sports", "Animation", "Action")
GROUP BY rental_month, category.name)

SELECT rental_month, category, rental_count,
LAG(rental_count) OVER(PARTITION BY category ORDER BY rental_month) as previous_month_count,
rental_count - LAG(rental_count) OVER(PARTITION BY category ORDER BY rental_month) as variation
FROM CTE


"""

run_query(query3)

,rental_month,category,rental_count,previous_month_count,variation
0,2005-05,Action,87,NaN,NaN
1,2005-06,Action,160,87.0,73.0
2,2005-07,Action,464,160.0,304.0
3,2005-08,Action,384,464.0,-80.0
4,2006-02,Action,17,384.0,-367.0
5,2005-05,Animation,74,NaN,NaN
6,2005-06,Animation,174,74.0,100.0
7,2005-07,Animation,489,174.0,315.0
8,2005-08,Animation,408,489.0,-81.0
9,2006-02,Animation,21,408.0,-387.0


#API

## Exercice API  - Géocodage d'adresse

**Objectif :**
Utiliser l'API [GeoCodage](https://geoservices.ign.fr/documentation/services/services-deprecies) pour récupérer les coordonnées GPS (latitude et longitude) d'une adresse donnée.

**Consignes :**
1. Implémenter une fonction `get_location(address: str) -> dict` qui prend en paramètre une adresse (chaîne de caractères).
2. Cette fonction doit interroger l'API via une requête HTTP GET et récupérer la réponse au format JSON.
3. Extraire et retourner un dictionnaire contenant au minimum :
   - 'adresse' : le nom complet normalisé du lieu retourné par l'API,
   - 'lat' : la latitude,
   - 'lon' : la longitude.
4. Gérer le cas où aucune donnée n'est retournée (retourner un dictionnaire vide).

Exemple d'adresse à tester : "5, rue Baron Nante"
"""


In [ ]:
import requests
from pprint import pprint

def get_location(address: str) -> dict:
    # Je définis mon url avec la stucture de data.gouv et avec le f string j'ajoute mon adresse entre accolade
    url = f"https://api-adresse.data.gouv.fr/search/?q={address}"
    # Je fais ma requête avec le .get et le transforme en .json avec du chaining, je le stocke dans ma variable result
    result = requests.get(url).json()
    # Je crée une variable coord pour récuperer les coordonnées de l'adresse
    coord = result['features'][0]['geometry']['coordinates'][::-1]
    # Je crée la variable label pour recuperer l'adresse
    label = result['features'][0]['properties']['label']
    # Je renvoie a travers un dictionnaire l'adresse et les coordonnées
    return {"adresse" : label, "lat" : coord[1], "lon" : coord[0]}

# Exemple d'utilisation
adresse = "5, rue Baron Nantes"
resultat = get_location(adresse)

print(resultat)


{'adresse': '5 Rue Baron 44000 Nantes', 'lat': -1.54768, 'lon': 47.211191}


In [ ]:
# Je regarde l'architecture de mon json pour m'aider a créer ma fonction

url = f"https://api-adresse.data.gouv.fr/search/?q={adresse}"
result = requests.get(url).json()
pprint(result)

{'features': [{'geometry': {'coordinates': [-1.54768, 47.211191],
                            'type': 'Point'},
               'properties': {'_type': 'address',
                              'banId': '774acc09-8234-409e-8e29-042d70cde561',
                              'city': 'Nantes',
                              'citycode': '44109',
                              'context': '44, Loire-Atlantique, Pays de la '
                                         'Loire',
                              'depcode': '44',
                              'housenumber': '5',
                              'id': '44109_0556_00005',
                              'importance': 0.72985,
                              'label': '5 Rue Baron 44000 Nantes',
                              'name': '5 Rue Baron',
                              'postcode': '44000',
                              'score': 0.8845318181818181,
                              'street': 'Rue Baron',
                              'type': 'house